<a href="https://colab.research.google.com/github/SathyaPrakashD/Classical-ML-Pipeline-Iris-Species-Classifier/blob/main/iris_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Classical ML Pipeline — Iris Species Classifier




## PURPOSE:
### Build a complete sklearn workflow from raw data to evaluated pipeline.
### This exercise consolidates 6 core sklearn modules covered in study:
###   1. Load built-in data
###   2. Split data correctly
###   3. Make data mathematically safe
###   4. Chain steps into one deployable object
###   5. Train a classifier
###   6. Measure how good the model actually is

## CHECKPOINT:
### By the end of this notebook, you will have a working ML pipeline that
### takes raw data and produces a classification_report — precision, recall,
### and F1 per class. A real, runnable ML workflow.

## Sklearn Import:


### We import exactly what we need — one tool from each core module.
### Nothing extra. Every import has a job.



In [2]:
from sklearn.datasets import load_iris              # built-in dataset
from sklearn.model_selection import train_test_split # split data correctly
from sklearn.preprocessing import StandardScaler    # scale numeric features
from sklearn.pipeline import Pipeline               # chain steps together
from sklearn.linear_model import LogisticRegression # our classifier
from sklearn.metrics import classification_report   # evaluate honestly

## Iris Dataset:

### The Iris dataset is built into sklearn — no file downloads needed.
### 150 samples. 4 numeric features. 3 target classes.
### X = input features (what the model learns FROM)
###     → sepal length, sepal width, petal length, petal width
### y = target labels (what the model learns TO predict)
###     → 0 = Setosa, 1 = Versicolour, 2 = Virginica

In [3]:
iris = load_iris()
X = iris.data    # shape: (150, 4) — 150 rows, 4 feature columns
y = iris.target  # shape: (150,)   — 150 labels

## Quick sanity check — confirm shapes and class names

In [4]:
print("Feature shape:", X.shape)
print("Target shape:", y.shape)
print("Classes:", iris.target_names)

Feature shape: (150, 4)
Target shape: (150,)
Classes: ['setosa' 'versicolor' 'virginica']


In [5]:
print((X))

[[5.1 3.5 1.4 0.2]
 [4.9 3.  1.4 0.2]
 [4.7 3.2 1.3 0.2]
 [4.6 3.1 1.5 0.2]
 [5.  3.6 1.4 0.2]
 [5.4 3.9 1.7 0.4]
 [4.6 3.4 1.4 0.3]
 [5.  3.4 1.5 0.2]
 [4.4 2.9 1.4 0.2]
 [4.9 3.1 1.5 0.1]
 [5.4 3.7 1.5 0.2]
 [4.8 3.4 1.6 0.2]
 [4.8 3.  1.4 0.1]
 [4.3 3.  1.1 0.1]
 [5.8 4.  1.2 0.2]
 [5.7 4.4 1.5 0.4]
 [5.4 3.9 1.3 0.4]
 [5.1 3.5 1.4 0.3]
 [5.7 3.8 1.7 0.3]
 [5.1 3.8 1.5 0.3]
 [5.4 3.4 1.7 0.2]
 [5.1 3.7 1.5 0.4]
 [4.6 3.6 1.  0.2]
 [5.1 3.3 1.7 0.5]
 [4.8 3.4 1.9 0.2]
 [5.  3.  1.6 0.2]
 [5.  3.4 1.6 0.4]
 [5.2 3.5 1.5 0.2]
 [5.2 3.4 1.4 0.2]
 [4.7 3.2 1.6 0.2]
 [4.8 3.1 1.6 0.2]
 [5.4 3.4 1.5 0.4]
 [5.2 4.1 1.5 0.1]
 [5.5 4.2 1.4 0.2]
 [4.9 3.1 1.5 0.2]
 [5.  3.2 1.2 0.2]
 [5.5 3.5 1.3 0.2]
 [4.9 3.6 1.4 0.1]
 [4.4 3.  1.3 0.2]
 [5.1 3.4 1.5 0.2]
 [5.  3.5 1.3 0.3]
 [4.5 2.3 1.3 0.3]
 [4.4 3.2 1.3 0.2]
 [5.  3.5 1.6 0.6]
 [5.1 3.8 1.9 0.4]
 [4.8 3.  1.4 0.3]
 [5.1 3.8 1.6 0.2]
 [4.6 3.2 1.4 0.2]
 [5.3 3.7 1.5 0.2]
 [5.  3.3 1.4 0.2]
 [7.  3.2 4.7 1.4]
 [6.4 3.2 4.5 1.5]
 [6.9 3.1 4.

In [6]:
print(y)

[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 2 2 2 2 2 2 2 2 2 2 2
 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2
 2 2]


##Split into Train and Test Sets

## Why Split:
#### The model must NEVER see test data during training.
#### test_size=0.2  → 80% train, 20% test
#### random_state=42 → same split every time you run (reproducible)

#### This gives us:
####   X_train, y_train → model learns from these (120 samples)
####   X_test, y_test   → model is evaluated on these (30 samples)

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y,
    test_size=0.2,
    random_state=42
)

print("Training samples:", X_train.shape[0])
print("Test samples:", X_test.shape[0])

Training samples: 120
Test samples: 30


##Build the Pipeline:


#### A Pipeline snaps preprocessing and model together into ONE object.
#### No manual fit/transform juggling. No risk of applying wrong statistics.

## Step 1 — StandardScaler
####   Rescales all 4 numeric features to mean=0, std=1
####   Why: the 4 features are on different scales — scaling levels the field


## Step 2 — LogisticRegression
####   Learns the relationship between the 4 scaled features and the 3 classes
####   max_iter=200: gives the algorithm enough rounds to converge

#### The pipeline handles fit/transform correctly automatically:
####   pipeline.fit()     → scaler learns from train data, model trains on scaled train data
####   pipeline.predict() → scaler transforms test data, model predicts on scaled test data

In [8]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  LogisticRegression(max_iter=200))
])

print("Pipeline created:")
print(pipeline)

Pipeline created:
Pipeline(steps=[('scaler', StandardScaler()),
                ('model', LogisticRegression(max_iter=200))])


##Train the Model


#### One line. The pipeline:
####   1. Fits the scaler on X_train (learns mean and std)
####   2. Transforms X_train using those statistics
####   3. Trains LogisticRegression on the scaled X_train and y_train

#### The model learns the pattern — which feature combinations map to which class.

In [9]:
pipeline.fit(X_train, y_train)

print("Training complete.")


Training complete.


##Make Predictions


#### One line. The pipeline:
####   1. Transforms X_test using the SAME statistics learned from training
####      (NOT refitted on test data — this is the critical rule)
####   2. Passes scaled X_test through the trained model
####   3. Returns predicted class labels

#### predictions contains: 0, 1, or 2 for each of the 30 test samples

In [10]:
predictions = pipeline.predict(X_test)

print("Predictions:", predictions)
print("Actual:     ", y_test)


Predictions: [1 0 2 1 1 0 1 2 1 1 2 0 0 0 0 1 2 1 1 2 0 2 0 2 2 2 2 2 0 0]
Actual:      [1 0 2 1 1 0 1 2 1 1 2 0 0 0 0 1 2 1 1 2 0 2 0 2 2 2 2 2 0 0]


##Evaluate the Model

#### classification_report gives us the full picture per class:

####   Precision — of all samples predicted as class X, how many actually were X? (did the net catch junk?)

####   Recall    — of all actual class X samples, how many did we correctly catch?
####               (did the net miss fish?)

####   F1 Score  — balance between precision and recall

####   Support   — how many actual samples of each class were in the test set

#### target_names maps 0,1,2 back to readable class names

In [11]:
print(classification_report(y_test, predictions, target_names=iris.target_names))



              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      1.00      1.00         9
   virginica       1.00      1.00      1.00        11

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30

